# Check 03 — Runtime Adapter

**Category:** Module smoke check (fast regression; companion to pytest, not a full tutorial).

**Purpose:** Prove `OpenAIAgentsRuntimeAdapter` healthcheck works and `run_turn` with `planned_tool_call` emits `TOOL_INTENT` (simulation path).

**Prerequisites:**
- Python **3.12+** with project deps installed (`pip install -r requirements.txt` from repo root)
- Kernel: project **`.venv`** (see `notebooks/README.md`)
- Run cells **top to bottom** (bootstrap cell sets `sys.path` automatically)
- **No API key** required — deterministic, in-process only

**Related tutorial:** `tutorial_02_openai_adapter.ipynb`

**Modules exercised:** `src/runtime/openai_agents_runtime`, `src/schemas/events`

**PASS means:** Health prints healthy; event stream includes `tool_intent`; `PASS: runtime adapter planned tool-intent path`.

**Troubleshooting:** If `nest_asyncio` is missing in Jupyter, `pip install nest-asyncio` (listed in `requirements.txt`). Live OpenAI calls are **not** required here.

In [1]:
import pathlib
import sys

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))
_contracts_src = _root / "packages" / "eXo_adapters" / "packages" / "exo-brain-core-contracts" / "src"
if _contracts_src.is_dir():
    sys.path.insert(0, str(_contracts_src))

from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter
from src.schemas.events import RuntimeEventType

In [2]:
import asyncio

async def _run_check():
    adapter = OpenAIAgentsRuntimeAdapter()
    handle = await adapter.start_session("sess_runtime_nb", {"agent_id": "runtime-nb"})
    assert handle.session_id == "sess_runtime_nb"

    health = await adapter.healthcheck()
    caps = adapter.get_capabilities()
    print("health:", health.state.value, health.reason)
    print("capabilities provider:", caps.provider_id)

    context = {
        "run_id": "run_runtime_nb",
        "job_id": "job_runtime_nb",
        "task_id": "task_runtime_nb",
        "agent_id": "agent_runtime_nb",
        "planned_tool_call": {
            "call_id": "tc_runtime_nb",
            "tool_name": "fake_tool",
            "arguments": {"x": 1},
            "risk_tier": "low",
            "is_state_changing": False,
        },
    }
    events = []
    async for event in adapter.run_turn("sess_runtime_nb", "hello", context):
        events.append(event)
        print(event.event_type.value, event.payload)

    assert any(e.event_type == RuntimeEventType.TOOL_INTENT for e in events)
    print("PASS: runtime adapter planned tool-intent path")

# Works in both async-native kernels and standard synchronous kernels
try:
    loop = asyncio.get_running_loop()
    import nest_asyncio
    nest_asyncio.apply()
    loop.run_until_complete(_run_check())
except RuntimeError:
    asyncio.run(_run_check())

health: healthy adapter-initialized
capabilities provider: openai
tool_intent {}
PASS: runtime adapter planned tool-intent path
